# 02 - Gold Layer: Heuristic Feature Engineering

In the Lakehouse Medallion architecture, the Gold Layer represents highly refined, business-level aggregated data ready for machine learning consumption. 

While Phase III of our architecture will utilize Deep Sequence Learning (preserving the 86-week temporal dimension), we must establish a strong, interpretable Tabular Baseline. To do this without losing the temporal "story," we will extract advanced heuristics from the time-series.

1. **Volatility & Shape Extraction:** Calculate Variance and Kurtosis to capture behavioral spikes (mega-prescribing events).
2. **Recent Kinematics (EWMA):** Calculate the Exponentially Weighted Moving Average, heavily weighting the final 12 weeks, as recent behavior is the strongest predictor of future intent.
3. **Cyclic Behavior (FFT):** Apply a Fast Fourier Transform to extract the dominant frequency, identifying doctors who prescribe in recurring cycles.
4. **Consolidation:** Merge these temporal heuristics with static demographics to create a single row per HCP.

In [3]:
import pandas as pd
import numpy as np
from scipy.stats import kurtosis
from scipy.fft import fft
import warnings

# Suppress warnings from scipy if it encounters arrays of all zeros
warnings.filterwarnings('ignore')

# ==========================================
# 1. Data Ingestion & Sequence Formatting
# ==========================================
print("Loading Silver Layer Parquet...")

df = pd.read_parquet('data/silver_layer_longitudinal.parquet')

print(f"Total rows loaded: {len(df):,}")

# CRITICAL MLOPS STEP: Temporal Sorting
# Time-series math (like EWMA and FFT) will fail catastrophically if the data is not chronological.
print("Sorting data chronologically to guarantee temporal integrity...")
df = df.sort_values(by=['NUEVO_ID', 'WEEK_ID']).reset_index(drop=True)

# Updated Core Metrics based on Spearman Correlation Matrix
core_metrics = ['UC_TRX', 'ORAL_TRX', 'BRAND1_NBRX', 'DETAILS', 'SAMPLES', 'DIRECTMAIL']

# Filter only metrics that exist in the dataframe
available_core_metrics = [col for col in core_metrics if col in df.columns]
print(f"Metrics selected for heuristic extraction: {available_core_metrics}")

Loading Silver Layer Parquet...
Total rows loaded: 1,800,066
Sorting data chronologically to guarantee temporal integrity...
Metrics selected for heuristic extraction: ['UC_TRX', 'ORAL_TRX', 'BRAND1_NBRX', 'DETAILS', 'SAMPLES', 'DIRECTMAIL']


In [ ]:
# ==========================================
# 2. Advanced Feature Extraction Functions
# ==========================================
print("Defining heuristic mathematical functions...")

def calculate_ewma(series, span=12):
    """
    Calculates the final value of an Exponentially Weighted Moving Average.
    A span of 12 heavily weights the last ~3 months of the 86-week sequence.
    This captures the 'momentum' of the HCP right before the prediction point.

    Edge-case handling:
      - Returns 0.0 for empty series, all-zero series, or series containing only NaNs.
      - Drops NaN values before computing to avoid NaN propagation.
    """
    clean = series.dropna()
    if len(clean) == 0 or clean.sum() == 0:
        return 0.0
    return clean.ewm(span=span, adjust=False).mean().iloc[-1]

def calculate_dominant_frequency(series):
    """
    Applies Fast Fourier Transform (FFT) to extract cyclic behavior.
    Ignores the DC component (index 0) which is just the sum of the signal.
    Only examines the first half of the FFT spectrum (the second half is a
    conjugate mirror for real-valued signals).
    Helps identify HCPs with recurring prescription/marketing cycles.

    Edge-case handling:
      - Returns 0.0 for series with fewer than 2 non-null values.
      - Returns 0.0 for all-zero or all-NaN series.
      - Fills NaN values with 0.0 before FFT (NaN breaks np.fft).
    """
    clean = series.fillna(0.0)
    if len(clean) < 2 or clean.sum() == 0:
        return 0.0

    vals = clean.values
    y_fft = np.abs(fft(vals))

    # Only consider the positive-frequency half of the spectrum (indices 1..N//2).
    # Index 0 is the DC component (signal mean) and is intentionally excluded.
    half = len(y_fft) // 2
    if half < 2:
        return 0.0

    return float(np.max(y_fft[1:half]))

def calculate_kurtosis(series):
    """
    Calculates Fisher's Kurtosis.
    High kurtosis (>3) indicates extreme, infrequent spikes (mega-prescribing events).
    Low kurtosis indicates consistent, flat behavior.

    Edge-case handling:
      - Returns 0.0 for series with fewer than 4 non-null values.
      - Returns 0.0 for all-zero or all-NaN series.
      - Returns 0.0 if variance is zero (constant series), which avoids
        NaN/Inf from scipy.stats.kurtosis division-by-zero.
    """
    clean = series.dropna()
    if len(clean) < 4 or clean.sum() == 0:
        return 0.0

    # Guard against constant (zero-variance) series
    if clean.std() == 0:
        return 0.0

    # Fisher=True subtracts 3 to make the normal distribution 0.0
    result = kurtosis(clean.values, fisher=True, bias=False)

    # Final guard: scipy may still return NaN in degenerate edge cases
    if np.isnan(result) or np.isinf(result):
        return 0.0

    return float(result)

print("Functions successfully loaded into memory.")


In [ ]:
# ==========================================
# 3. Mass Feature Extraction Engine
# ==========================================
print("Executing Heuristic Extraction Engine across all HCPs...")
print("Note: This step performs heavy mathematics (FFT, Kurtosis) and might take a few minutes.\n")

# List to store processed dataframes for each metric
processed_features = []

# Group by HCP ID (single groupby, reused across all metrics)
grouped = df.groupby('NUEVO_ID')

for metric in available_core_metrics:
    print(f"Processing metric: {metric}...")

    # Apply our heuristic functions
    agg_df = grouped[metric].agg(
        MEAN='mean',
        EWMA=calculate_ewma,
        FFT=calculate_dominant_frequency,
        KURTOSIS=calculate_kurtosis
    ).reset_index()

    # Rename columns to clearly identify the parent metric (e.g., 'UC_TRX_FFT')
    agg_df = agg_df.rename(columns={
        'MEAN': f'{metric}_MEAN',
        'EWMA': f'{metric}_EWMA',
        'FFT': f'{metric}_FFT',
        'KURTOSIS': f'{metric}_KURTOSIS'
    })

    processed_features.append(agg_df)

# Merge all processed metrics back into a single tabular dataframe
print("\nMerging heuristic features into a single tabular view...")
gold_features = processed_features[0]
for i in range(1, len(processed_features)):
    gold_features = pd.merge(gold_features, processed_features[i], on='NUEVO_ID', how='inner')

# ==========================================
# 4. Static Demographics & Label Preservation
# ==========================================
print("Extracting static demographics and ground-truth labels...")

# Define the static columns we want to preserve
static_cols = ['SPECIALTY', 'IS_LABELED', 'ATSEG']
available_static = [col for col in static_cols if col in df.columns]

# Extract the first chronological instance of these static traits per HCP
static_df = df.groupby('NUEVO_ID')[available_static].first().reset_index()

# Final Merge: Combine Demographics with Temporal Heuristics
gold_layer = pd.merge(static_df, gold_features, on='NUEVO_ID', how='inner')

# ==========================================
# Data Integrity Assertion: 1 HCP = 1 Row
# ==========================================
expected_hcp_count = df['NUEVO_ID'].nunique()
actual_row_count = len(gold_layer)
duplicate_ids = gold_layer['NUEVO_ID'].duplicated().sum()

assert duplicate_ids == 0, (
    f"CRITICAL: Duplicate HCP rows detected after merge! "
    f"Found {duplicate_ids} duplicates. Pipeline integrity compromised."
)

assert actual_row_count == expected_hcp_count, (
    f"WARNING: Row count mismatch. Expected {expected_hcp_count} HCPs, "
    f"got {actual_row_count}. Some HCPs may have been dropped during inner merge."
)

print(f"\nData Integrity Check PASSED: {actual_row_count} rows, 0 duplicates.")

# ==========================================
# 5. Pipeline Export
# ==========================================
output_file = 'data/gold_heuristic_features.parquet'
gold_layer.to_parquet(output_file, index=False)

print("\n==================================================")
print("SUCCESS: Gold Layer Tabular Baseline Generated!")
print("==================================================")
print(f"Final Data Shape: {gold_layer.shape} (Should be exactly 1 row per HCP)")
print(f"File saved as: '{output_file}' ready for Phase IV (XGBoost).")


## Step 5: Gold Layer Validation & Sign-off

Before this dataset moves to Phase IV (XGBoost Ensemble), the following automated validations must pass. This cell serves as the **quality gate** of the Gold Layer pipeline.


In [ ]:
# ==========================================
# Gold Layer Validation & Sign-off Report
# ==========================================
print("=" * 60)
print("   GOLD LAYER QUALITY GATE — AUTOMATED VALIDATION REPORT")
print("=" * 60)

# --- 1. Dimensional Integrity ---
n_rows, n_cols = gold_layer.shape
n_unique_hcps = gold_layer['NUEVO_ID'].nunique()
n_expected_hcps = df['NUEVO_ID'].nunique()

print(f"\n[1] DIMENSIONAL INTEGRITY")
print(f"    Total rows:          {n_rows:,}")
print(f"    Total columns:       {n_cols}")
print(f"    Unique HCP IDs:      {n_unique_hcps:,}")
print(f"    Expected HCP count:  {n_expected_hcps:,}")

dim_ok = (n_rows == n_unique_hcps == n_expected_hcps)
print(f"    1-HCP-per-row check: {'PASS ✓' if dim_ok else 'FAIL ✗'}")

# --- 2. Duplicate Detection ---
n_dup = gold_layer['NUEVO_ID'].duplicated().sum()
print(f"\n[2] DUPLICATE DETECTION")
print(f"    Duplicate NUEVO_ID:  {n_dup}")
print(f"    Status:              {'PASS ✓' if n_dup == 0 else 'FAIL ✗'}")

# --- 3. NaN / Null Audit ---
nan_report = gold_layer.isnull().sum()
total_nans = nan_report.sum()
cols_with_nans = nan_report[nan_report > 0]

print(f"\n[3] NaN / NULL AUDIT")
print(f"    Total NaN values:    {total_nans}")

if len(cols_with_nans) > 0:
    print(f"    Columns with NaNs:")
    for col, cnt in cols_with_nans.items():
        print(f"      - {col}: {cnt} NaN(s) ({cnt/n_rows*100:.2f}%)")
    nan_ok = False
else:
    print(f"    Columns with NaNs:   None")
    nan_ok = True
print(f"    Status:              {'PASS ✓' if nan_ok else 'REVIEW REQUIRED ⚠'}")

# --- 4. Infinity Check ---
numeric_cols = gold_layer.select_dtypes(include='number').columns
inf_count = np.isinf(gold_layer[numeric_cols]).sum().sum()
print(f"\n[4] INFINITY CHECK")
print(f"    Infinite values:     {inf_count}")
print(f"    Status:              {'PASS ✓' if inf_count == 0 else 'FAIL ✗'}")

# --- 5. Feature Summary ---
heuristic_cols = [c for c in gold_layer.columns if c not in ['NUEVO_ID'] + available_static]
print(f"\n[5] FEATURE INVENTORY")
print(f"    Static features:     {available_static}")
print(f"    Heuristic features:  {len(heuristic_cols)}")
print(f"    Feature names:       {heuristic_cols}")

# --- Final Verdict ---
all_pass = dim_ok and (n_dup == 0) and nan_ok and (inf_count == 0)
print(f"\n{'=' * 60}")
if all_pass:
    print("   VERDICT: ALL CHECKS PASSED ✓")
    print("   Gold Layer is APPROVED for Phase IV (XGBoost Ensemble).")
else:
    print("   VERDICT: SOME CHECKS FAILED ✗")
    print("   Gold Layer requires manual review before proceeding.")
print(f"{'=' * 60}")
